In [1]:
import json
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
with open('notebooks/Arabic_experiments/ArabicSocialMediaDataset/cross_model_detection_multiple_runs_results.json', 'r') as f:
    data = json.load(f)

In [3]:
results = {}
metrics = ['accuracy', 'precision', 'recall', 'f1']

In [4]:
for train_model in data.keys():
    results[train_model] = {}
    
    # For each test model
    for test_model in ['allam', 'jais-batched', 'llama-batched', 'openai']:
        results[train_model][test_model] = {}
        
        # Extract values for each metric across all runs
        for metric in metrics:
            values = []
            for run in data[train_model]:
                if test_model in run:
                    values.append(run[test_model][metric] * 100)  # Convert to percentage
            
            # Calculate mean and std
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1)  # Sample standard deviation
            
            results[train_model][test_model][metric] = {
                'mean': mean_val,
                'std': std_val
            }

In [5]:
def create_summary_table(results, with_std=True):
    """
    Create a summary table for cross-model performance.
    
    Parameters:
    results (dict): Results dictionary from analyze_cross_model_performance
    with_std (bool): Whether to include standard deviation in the output
    """
    
    summary_data = []
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    
    # Header
    if with_std:
        header = ['Metric', 'Train → Test', 'ALLaM', 'Jais', 'Llama', 'OpenAI']
    else:
        header = ['Metric', 'Train → Test', 'ALLaM', 'Jais', 'Llama', 'OpenAI']
    
    model_display_names = {
        'allam': 'ALLaM',
        'jais-batched': 'Jais',
        'llama-batched': 'Llama', 
        'openai': 'OpenAI'
    }
    
    test_models = ['allam', 'jais-batched', 'llama-batched', 'openai']
    
    for metric in metrics:
        for i, train_model in enumerate(['allam', 'jais-batched', 'llama-batched', 'openai']):
            if i == 0:  # First row for this metric
                row = [metric.capitalize(), model_display_names[train_model]]
            else:
                row = ['', model_display_names[train_model]]
            
            for test_model in test_models:
                mean_val = results[train_model][test_model][metric]['mean']
                std_val = results[train_model][test_model][metric]['std']
                
                if with_std and std_val > 0:
                    # Only show std if it's greater than 0
                    formatted_value = f"{mean_val:.2f} ± {std_val:.2f}"
                else:
                    formatted_value = f"{mean_val:.2f}"
                
                row.append(formatted_value)
            
            summary_data.append(row)
    
    return tabulate(summary_data, headers=header, tablefmt='grid', stralign='center')

In [6]:
print(create_summary_table(results=results))

+-----------+----------------+--------------+--------------+--------------+--------------+
|  Metric   |  Train → Test  |    ALLaM     |     Jais     |    Llama     |    OpenAI    |
+===========+================+==============+==============+==============+==============+
| Accuracy  |     ALLaM      | 92.54 ± 0.69 | 96.14 ± 0.53 | 76.23 ± 0.81 | 97.32 ± 0.55 |
+-----------+----------------+--------------+--------------+--------------+--------------+
|           |      Jais      | 88.54 ± 0.68 | 96.97 ± 0.51 | 73.44 ± 1.60 | 93.31 ± 1.66 |
+-----------+----------------+--------------+--------------+--------------+--------------+
|           |     Llama      | 65.41 ± 0.26 | 64.09 ± 0.53 | 98.80 ± 0.20 | 60.54 ± 0.42 |
+-----------+----------------+--------------+--------------+--------------+--------------+
|           |     OpenAI     | 89.21 ± 1.09 | 94.76 ± 0.85 | 72.33 ± 1.87 | 98.90 ± 0.29 |
+-----------+----------------+--------------+--------------+--------------+--------------+